# Análisis de Ratings Elo - Mundial 2026

In [2]:
import pandas as pd
import numpy as np
from collections import defaultdict
import matplotlib.pyplot as plt
import plotly.express as px
import os

os.makedirs('output', exist_ok=True)

## Cargar datos

In [3]:
matches = pd.read_csv(r"Data\Matches.csv")
matches['date'] = pd.to_datetime(matches['date'])
matches=matches.dropna()
print(f"Partidos: {len(matches)}")
print(f"Fechas: {matches['date'].min()} a {matches['date'].max()}")

Partidos: 7952
Fechas: 2018-01-02 00:00:00 a 2026-03-31 00:00:00


## Calcular Elo (K variable)

In [4]:
matches["tournament"].value_counts()

tournament
Friendly                                2135
FIFA World Cup qualification            1767
UEFA Nations League                      658
African Cup of Nations qualification     560
UEFA Euro qualification                  501
                                        ... 
Marianas Cup                               2
Copa América qualification                 2
Mukuru 4 Nations                           2
South Asian Super Cup                      1
CONMEBOL–UEFA Cup of Champions             1
Name: count, Length: 71, dtype: int64

In [5]:
# Diccionario K por torneo
K_VALUES = {
    # Mundial
    'FIFA World Cup': 60,
    # Clasificatorias
    'FIFA World Cup qualification': 25,
    # Europa
    'UEFA Euro': 35,
    'UEFA Euro qualification': 20,
    'UEFA Nations League': 15,
    # Sudamérica
    'Copa América': 35,
    # Amistosos
    'Friendly': 5,
}

# Asignar K
matches['k_factor'] = matches['tournament'].map(K_VALUES).fillna(0)

# Revisar valores
matches[['tournament', 'k_factor']].drop_duplicates().sort_values('k_factor', ascending=False)

,tournament,k_factor
342,FIFA World Cup,60.0
1303,Copa América,35.0
2784,UEFA Euro,35.0
1187,FIFA World Cup qualification,25.0
1001,UEFA Euro qualification,20.0
...,...,...
2863,Arab Cup qualification,0.0
2903,Gold Cup qualification,0.0
3033,Three Nations Cup,0.0
3473,Arab Cup,0.0


In [6]:
def calculate_elo(matches_df, initial_rating=1500, home_adv=0):
    """Calcula ratings Elo para todos los equipos."""
    ratings = defaultdict(lambda: initial_rating)
    history = []
    matches_df = matches_df.sort_values('date')

    for _, row in matches_df.iterrows():
        home, away = row['home_team'], row['away_team']
        home_goals, away_goals = row['home_score'], row['away_score']
        k = row['k_factor']

        home_elo, away_elo = ratings[home], ratings[away]

        # Probabilidad esperada
        expected_home = 1 / (1 + 10 ** ((away_elo - home_elo - home_adv) / 400))
        expected_away = 1 - expected_home

        # Resultado real
        if home_goals > away_goals:
            result_home, result_away = 1, 0
        elif home_goals < away_goals:
            result_home, result_away = 0, 1
        else:
            result_home, result_away = 0.5, 0.5

        # Actualizar ratings
        ratings[home] += k * (result_home - expected_home)
        ratings[away] += k * (result_away - expected_away)

        # Guardar historial
        history.append({
            'date': row['date'],
            'home_team': home,
            'away_team': away,
            'home_score': home_goals,
            'away_score': away_goals,
            'home_elo_before': home_elo,
            'away_elo_before': away_elo,
            'home_elo_after': ratings[home],
            'away_elo_after': ratings[away],
            'k_factor': k
        })

    return dict(ratings), pd.DataFrame(history)

In [7]:
elo_championship, elo_history = calculate_elo(matches)

print(f"Equipos únicos: {len(elo_championship)}")

Equipos únicos: 282


## Top 20 - Elo de Campeonato

In [8]:
elo_championship_df = pd.DataFrame(list(elo_championship.items()), columns=['Equipo', 'Rating'])
elo_championship_df = elo_championship_df.sort_values('Rating', ascending=False).reset_index(drop=True)
print("=== Top 20 Equipos - Elo Campeonato ===\n")
print(elo_championship_df.head(20).to_string())

=== Top 20 Equipos - Elo Campeonato ===

         Equipo       Rating
0         Spain  1807.709340
1        France  1775.751136
2     Argentina  1753.136721
3       England  1749.580139
4   Netherlands  1707.512013
5         Japan  1704.299788
6      Portugal  1692.415652
7       Germany  1679.939508
8       Morocco  1678.319761
9       Croatia  1670.307297
10       Brazil  1662.945175
11  South Korea  1662.809733
12         Iran  1659.267947
13      Belgium  1658.803875
14        Italy  1656.505429
15       Turkey  1654.080533
16    Australia  1653.390723
17       Norway  1646.153813
18  Switzerland  1640.991848
19      Tunisia  1639.712630


In [9]:
elo_championship_df

,Equipo,Rating
0,Spain,1807.709340
1,France,1775.751136
2,Argentina,1753.136721
3,England,1749.580139
4,Netherlands,1707.512013
...,...,...
277,Lithuania,1316.962793
278,Andorra,1273.329509
279,Gibraltar,1244.672224
280,Liechtenstein,1208.161743


## Elo de Shootouts (Penales)

In [10]:
shootouts = pd.read_csv('Data/shootouts.csv')
shootouts['date'] = pd.to_datetime(shootouts['date'])
shootouts = shootouts[shootouts['date'] >= pd.to_datetime('2000-01-01')]
print(f"Shootouts desde junio 2022: {len(shootouts)}")

Shootouts desde junio 2022: 413


In [11]:
valid_teams = set(matches["home_team"]) | set(matches["away_team"])

shootouts = shootouts[
    shootouts["home_team"].isin(valid_teams) &
    shootouts["away_team"].isin(valid_teams)
]

In [12]:
# Configuración para shootouts
K = 40
FIRST_BONUS = 30

elo_shootout = defaultdict(lambda: 1500)


def expected(r1, r2):
    return 1 / (1 + 10 ** ((r2 - r1) / 400))


for _, row in shootouts.iterrows():
    home, away = row["home_team"], row["away_team"]
    winner, first = row["winner"], row["first_shooter"]

    r_home, r_away = elo_shootout[home], elo_shootout[away]

    # Ventaja por empezar
    if pd.notna(first):
        if first == home:
            r_home += FIRST_BONUS
        elif first == away:
            r_away += FIRST_BONUS

    exp_home = expected(r_home, r_away)
    exp_away = 1 - exp_home

    # Resultado real
    s_home = 1 if winner == home else 0
    s_away = 1 - s_home

    # Actualizar
    elo_shootout[home] += K * (s_home - exp_home)
    elo_shootout[away] += K * (s_away - exp_away)

In [13]:
elo_shootout_df = pd.DataFrame(
    elo_shootout.items(),
    columns=["Team", "Elo Pen"]
)

elo_shootout_df = elo_shootout_df.sort_values(
    by="Elo Pen",
    ascending=False
).reset_index(drop=True)

print(elo_shootout_df.head(40))

                      Team      Elo Pen
0                    Qatar  1580.540733
1                  Padania  1573.742925
2                     Iraq  1572.045259
3                   Zambia  1561.790484
4                     Oman  1558.755288
5                 Ethiopia  1557.771431
6               Kazakhstan  1557.263926
7                  Senegal  1557.111520
8               Seychelles  1556.891174
9                   Panama  1548.144777
10                 Algeria  1546.494771
11             South Korea  1546.225330
12                Botswana  1544.694925
13                  Jersey  1544.332930
14                Honduras  1544.276240
15                   Chile  1543.876777
16    United Arab Emirates  1542.022550
17                 Iceland  1541.908231
18               Argentina  1541.479894
19                 Germany  1540.883503
20              Kárpátalja  1539.985949
21                  Tahiti  1539.939724
22               Greenland  1539.934031
23                  Cyprus  1538.849977


## Comparación

In [14]:
# Unir Elo general y Elo de shootout
combined = elo_championship_df.merge(elo_shootout_df, left_on="Equipo", right_on="Team", how="outer")
combined = combined.fillna(1500)
combined["Diff"] = combined["Rating"] - combined["Elo Pen"]
combined = combined.sort_values(by="Rating", ascending=False).reset_index(drop=True)

print("=== Comparación Elo General vs Elo Shootout ===\n")
print(combined.head(20).to_string(index=False))

=== Comparación Elo General vs Elo Shootout ===

     Equipo      Rating        Team     Elo Pen       Diff
      Spain 1807.709340       Spain 1510.569337 297.140003
     France 1775.751136      France 1480.249579 295.501556
  Argentina 1753.136721   Argentina 1541.479894 211.656827
    England 1749.580139     England 1489.407155 260.172983
Netherlands 1707.512013 Netherlands 1457.061724 250.450289
      Japan 1704.299788       Japan 1463.306187 240.993601
   Portugal 1692.415652    Portugal 1536.203137 156.212516
    Germany 1679.939508     Germany 1540.883503 139.056006
    Morocco 1678.319761     Morocco 1491.681425 186.638336
    Croatia 1670.307297     Croatia 1526.279239 144.028058
     Brazil 1662.945175      Brazil 1515.620578 147.324597
South Korea 1662.809733 South Korea 1546.225330 116.584403
       Iran 1659.267947        Iran 1464.854103 194.413845
    Belgium 1658.803875        1500 1500.000000 158.803875
      Italy 1656.505429       Italy 1526.515442 129.989988
     Tu

In [16]:
# Unir Elo general y Elo de penales
combined = elo_championship_df.merge(
    elo_shootout_df,
    left_on="Equipo",
    right_on="Team",
    how="outer"
)

# Reemplazar valores faltantes con 1500
combined = combined.fillna(1500)

# Diferencia entre ambos ratings
combined["Diff"] = (
    combined["Rating"] -
    combined["Elo Pen"]
)

# Ordenar por Elo general
combined = combined.sort_values(
    by="Rating",
    ascending=False
).reset_index(drop=True)

# Mostrar top 20
print("=== Comparación Elo General vs Elo Penales ===\n")

print(
    combined.head(20).to_string(index=False)
)

=== Comparación Elo General vs Elo Penales ===

     Equipo      Rating        Team     Elo Pen       Diff
      Spain 1807.709340       Spain 1510.569337 297.140003
     France 1775.751136      France 1480.249579 295.501556
  Argentina 1753.136721   Argentina 1541.479894 211.656827
    England 1749.580139     England 1489.407155 260.172983
Netherlands 1707.512013 Netherlands 1457.061724 250.450289
      Japan 1704.299788       Japan 1463.306187 240.993601
   Portugal 1692.415652    Portugal 1536.203137 156.212516
    Germany 1679.939508     Germany 1540.883503 139.056006
    Morocco 1678.319761     Morocco 1491.681425 186.638336
    Croatia 1670.307297     Croatia 1526.279239 144.028058
     Brazil 1662.945175      Brazil 1515.620578 147.324597
South Korea 1662.809733 South Korea 1546.225330 116.584403
       Iran 1659.267947        Iran 1464.854103 194.413845
    Belgium 1658.803875        1500 1500.000000 158.803875
      Italy 1656.505429       Italy 1526.515442 129.989988
     Tur

In [17]:
combined[combined["Equipo"]== "Chile"]

,Equipo,Rating,Team,Elo Pen,Diff
238,Chile,1435.276497,Chile,1543.876777,-108.60028


## Evolución - Un Equipo

In [18]:
elo_history.head(3)

,date,home_team,away_team,home_score,away_score,home_elo_before,away_elo_before,home_elo_after,away_elo_after,k_factor
0,2018-01-02,Iraq,United Arab Emirates,0.0,0.0,1500.0,1500.0,1500.0,1500.0,0.0
1,2018-01-02,Oman,Bahrain,1.0,0.0,1500.0,1500.0,1500.0,1500.0,0.0
2,2018-01-05,Oman,United Arab Emirates,0.0,0.0,1500.0,1500.0,1500.0,1500.0,0.0


In [19]:
# Evolución de un equipo
team = "Germany"

# Partidos como local
home_games = elo_history[elo_history["home_team"] == team][["date", "home_elo_after"]].copy()
home_games.columns = ["date", "elo"]

# Partidos como visitante
away_games = elo_history[elo_history["away_team"] == team][["date", "away_elo_after"]].copy()
away_games.columns = ["date", "elo"]

# Unir historial
team_history = pd.concat([home_games, away_games]).sort_values("date")
team_history["date"] = pd.to_datetime(team_history["date"])

# Gráfico interactivo
fig = px.line(team_history, x="date", y="elo", title=f"Evolución Elo - {team}", markers=True)
fig.update_layout(xaxis_title="Fecha", yaxis_title="Elo", hovermode="x unified", template="plotly_white")
fig.show()

## Evolución - Varios Equipos

In [ ]:
# Evolución de varios equipos
teams = ["Peru", "Iran", "Senegal"]
all_history = []

for team in teams:
    # Partidos como local
    home_games = elo_history[elo_history["home_team"] == team][["date", "home_elo_after"]].copy()
    home_games.columns = ["date", "elo"]
    
    # Partidos como visitante
    away_games = elo_history[elo_history["away_team"] == team][["date", "away_elo_after"]].copy()
    away_games.columns = ["date", "elo"]
    
    # Unir y agregar nombre
    team_history = pd.concat([home_games, away_games])
    team_history["team"] = team
    all_history.append(team_history)

# Unir todos y ordenar
history_df = pd.concat(all_history)
history_df["date"] = pd.to_datetime(history_df["date"])
history_df = history_df.sort_values("date")

# Gráfico interactivo
fig = px.line(history_df, x="date", y="elo", color="team", title="Evolución Elo - Principales Candidatos", markers=False)
fig.update_layout(xaxis_title="Fecha", yaxis_title="Elo", hovermode="x unified", template="plotly_white")
fig.show()

## Guardar CSVs

In [20]:
# Guardar archivos
elo_championship_df.to_csv("output/ratings_championship.csv", index=False)
elo_shootout_df.to_csv("output/ratings_shootout.csv", index=False)
combined.to_csv("output/ratings_combined.csv", index=False)
elo_history.to_csv("output/elo_history.csv", index=False)

print("Archivos guardados en output/")

Archivos guardados en output/


In [21]:
import pandas as pd
import numpy as np
from scipy.stats import poisson
import matplotlib.pyplot as plt
import random
from collections import Counter

# Cargar ratings
ratings_df = pd.read_csv('output/ratings_championship.csv')
print("Ratings cargados:")
print(ratings_df.head(10))

Ratings cargados:
        Equipo       Rating
0        Spain  1807.709340
1       France  1775.751136
2    Argentina  1753.136721
3      England  1749.580139
4  Netherlands  1707.512013
5        Japan  1704.299788
6     Portugal  1692.415652
7      Germany  1679.939508
8      Morocco  1678.319761
9      Croatia  1670.307297
